In [ ]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd

import torch
import wandb

from matplotlib.patches import Patch

from hydra import initialize, initialize_config_module, initialize_config_dir, compose
from omegaconf import OmegaConf

import lightning as lit
from datetime import datetime

from machine_intelligence.data.SepsisPredDataModule import SepsisPredDataModule

from machine_intelligence.models.GatedTransformer import GatedTransformer

from machine_intelligence.utilities.viz_setup import figure_setup_grid, VariableNames, insert_line_breaks_in_columns
from machine_intelligence.utilities.viz_setup import figure_setup, five_min_to_hours
from machine_intelligence.utilities.visualizations_interpretability import shap_heatmap_mod, shap_heatmap_values

import shap


In [ ]:
cwd = Path.cwd()
cwd


In [ ]:
project_root = Path(cwd).parents[1]
project_root


In [ ]:
current_date = datetime.now().strftime("%Y-%m-%d")
plot_output_dir = project_root / "plot_outputs" / current_date
plot_output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
model_path = project_root / Path("machine_intelligence/models/trained_models/gtn_final-3/checkpoints/last.ckpt")
model = GatedTransformer.load_from_checkpoint(model_path, map_location="cpu")
# model = GatedTransformer.load_from_checkpoint(model_path)



In [ ]:
with initialize(version_base=None, config_path="../../configs"):
    cfg = compose(config_name="run_training.yaml",
                  overrides=["data=cohort_a_r0.4_prev_0.3_imp_none", "params.batch_size_gpu=400"])

    print(cfg)

In [ ]:
wandb.init(mode="disabled")

In [ ]:
os.environ['PYTHONHASHSEED'] = str(42)
lit.seed_everything(42, workers=True)
torch.use_deterministic_algorithms(True)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
torch.backends.cudnn.benchmark = False
# os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"


In [ ]:
sepsis_pred_data = SepsisPredDataModule(cfg=cfg)


In [ ]:
sepsis_pred_data.setup(stage="fit")
sepsis_pred_data.setup(stage="test")

background_dl = sepsis_pred_data.train_dataloader()
# background_dl = sepsis_pred_data.val_dataloader() # the val dataloader doesn't "suffer" from shuffling -> reproducible results
inference_dl = sepsis_pred_data.test_dataloader()

In [ ]:
sepsis_pred_data.batch_size

In [ ]:
batch_background = next(iter(background_dl))

# batch = next(iter(inference_dl))
batch_features_ts, batch_features_stat, batch_labels = batch_background


In [ ]:
shap_device = "cuda"
# shap_device = "cpu"
# background_samples = [batch_features_ts, batch_features_stat]

background_samples = [batch_features_ts.to(shap_device), batch_features_stat.to(shap_device)]


In [ ]:
batch_explain = next(iter(inference_dl))

In [ ]:

batch_explain_ts, batch_explain_stat, batch_explain_labels = batch_explain

explain_samples = [batch_explain_ts.to(shap_device), batch_explain_stat.to(shap_device)]

In [ ]:
model.to(shap_device)
model.eval()

with torch.no_grad():
    pred, enc, score_input, score_input_sm, score_channel, score_channel_sm, gather_input, gather_channel, gate = model(
        batch_explain_ts.to(shap_device), batch_explain_stat.to(shap_device), stage="test")



In [ ]:
class ModelShapWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        # self.sigmoid = torch.nn.Sigmoid()

    def forward(self, x_ts_in, x_static_in):
        # self.model.eval()

        if isinstance(self.model, GatedTransformer):
            x, _, _, _, _, _, _, _, _ = self.model(x_ts_in, x_static_in, "test")
        else:
            x = self.model(x_ts_in, x_static_in)

        x = torch.nn.functional.sigmoid(x)

        return x



In [ ]:
model_shap = ModelShapWrapper(model=model)

In [ ]:
len(explain_samples[:2])

In [ ]:
explainer = shap.GradientExplainer(
    model=model_shap,
    # data=background_samples[:2],
    data=background_samples,
)

In [ ]:
explainer.feature_names = sepsis_pred_data.considered_features


In [ ]:
shap_values = explainer.shap_values(explain_samples[:2])



In [ ]:
df_explanations_ts = pd.DataFrame(
    data=explain_samples[0].to("cpu").reshape(-1, len(sepsis_pred_data.considered_features)),
    columns=sepsis_pred_data.considered_features)


In [ ]:
df_explanations_ts.head()

In [ ]:
shap_explanation_ts = shap.Explanation(shap_values[0],
                                       # base_values=explainer.expected_value,
                                       data=df_explanations_ts,
                                       feature_names=df_explanations_ts.columns)

In [ ]:
shap_explanation_ts.shape

In [ ]:
from tqdm.auto import tqdm

shap_values_batches = []
sample_ts_batches = []
# with torch.no_grad():   already using eval mode
for batch in tqdm(inference_dl, desc="Calculating shap values"):
    sample_ts, sample_stat, sample_label = batch

    sample_ts = sample_ts.to(shap_device)
    sample_stat = sample_stat.to(shap_device)

    shap_values = explainer.shap_values([sample_ts, sample_stat],
                                        # check_additivity=False
                                        )

    shap_values_batches.append(shap_values)
    sample_ts_batches.append(sample_ts)


In [ ]:
shap_values_batches_flat_ts = [ndarray for sublist in shap_values_batches for ndarray in sublist[0]]
len(shap_values_batches_flat_ts)  # should be the number of total samples, e.g. 1529 for r0.4

In [ ]:

shap_values_batches_flat_static = [ndarray for sublist in shap_values_batches for ndarray in sublist[1]]
len(shap_values_batches_flat_static)  # should be the number of total samples, e.g. 1529 for r0.4


In [ ]:
np.concatenate(shap_values_batches_flat_static, axis=1).shape

In [ ]:
shap_values_batches_flat_ts[0].shape

In [ ]:
shap_values_test_ts_mean = np.absolute(np.concatenate(shap_values_batches_flat_ts, axis=0).squeeze(2)).mean(axis=0)
shap_values_test_static_mean = np.absolute(np.concatenate(shap_values_batches_flat_static, axis=1)).mean(axis=1)


In [ ]:
shap_values_test_ts_mean.shape


In [ ]:
shap_values_test_static_mean.shape

In [ ]:
df_shap_values_ts_mean = pd.DataFrame(data=[shap_values_test_ts_mean], columns=sepsis_pred_data.considered_features)

In [ ]:
fig_ts_mean, ax_ts_mean, _, _ = figure_setup(1, 1, width_cm=14, height_cm=10)
sns.barplot(df_shap_values_ts_mean, ax=ax_ts_mean)

ax_ts_mean.set_xticklabels(ax_ts_mean.get_xticklabels(), fontsize=4, rotation=90)


## SHAP Values Confusion Matrix

In [ ]:


results = []

inference_idx = 0

for batch in tqdm(inference_dl, desc="Determining"):
    sample_ts, sample_stat, sample_label = batch

    sample_ts = sample_ts.to(shap_device)
    sample_stat = sample_stat.to(shap_device)

    outputs = model_shap(sample_ts, sample_stat)

    threshold = 0.5  # threshold may vary
    preds = (outputs >= threshold).int()

    for i in range(len(sample_label)):
        results.append({
            "Inference Index": inference_idx,
            "Prediction": "sepsis" if preds[i].item() == 1 else "control",
            "Ground Truth": "sepsis" if sample_label[i].item() == 1 else "control"
        })
        inference_idx += 1



In [ ]:
df_test_inference_results = pd.DataFrame(results)
df_test_inference_results

In [ ]:
shap_values_batches_flat_ts


In [ ]:
shap_values_batches_flat_ts_stacked = np.stack(shap_values_batches_flat_ts)
shap_values_batches_flat_ts_stacked.shape

In [ ]:
shap_values_batches_flat_static_stacked = np.stack(shap_values_batches_flat_static)
shap_values_batches_flat_static_stacked.shape

we don't need to average the shap values for the static like we did for the ts shap values, as the barplot function will do this for us anyway

In [ ]:
shap_values_ts_all = np.mean(np.absolute(shap_values_batches_flat_ts_stacked), axis=1).squeeze(2)
# shap_values_static_all = np.mean(np.absolute(shap_values_batches_flat_static_stacked), axis=0)
shap_values_static_all = np.absolute(shap_values_batches_flat_static_stacked).squeeze(2)
shap_values_static_all.shape

In [ ]:
df_shap_ts_all = pd.DataFrame(data=shap_values_ts_all, columns=sepsis_pred_data.considered_features)
df_shap_ts_all.head()

In [ ]:
df_shap_static_all = pd.DataFrame(data=shap_values_static_all, columns=sepsis_pred_data.considered_features_static)
df_shap_static_all.head()


In [ ]:
df_shap_ts_all_with_results = pd.concat([df_test_inference_results.drop(columns=["Inference Index"]), df_shap_ts_all],
                                        axis=1)
df_shap_ts_all_with_results.head()

In [ ]:
df_shap_static_all_with_results = pd.concat(
    [df_test_inference_results.drop(columns=["Inference Index"]), df_shap_static_all],
    axis=1)
df_shap_static_all_with_results.head()

# Time Series SHAP "confusion matrix" barplots

Time Series Confusion SHAP values 0.5 cut off

In [ ]:
df_shap_ts_gt_control = df_shap_ts_all_with_results.loc[df_shap_ts_all_with_results["Ground Truth"] == "control",
                        :].drop(columns=["Ground Truth", "Prediction"])

df_shap_ts_gt_sepsis = df_shap_ts_all_with_results.loc[df_shap_ts_all_with_results["Ground Truth"] == "sepsis", :].drop(
    columns=["Ground Truth", "Prediction"])

df_shap_ts_pred_control = df_shap_ts_all_with_results.loc[df_shap_ts_all_with_results["Prediction"] == "control",
                          :].drop(columns=["Ground Truth", "Prediction"])

df_shap_ts_pred_sepsis = df_shap_ts_all_with_results.loc[df_shap_ts_all_with_results["Prediction"] == "sepsis",
                         :].drop(columns=["Ground Truth", "Prediction"])

In [ ]:
df_shap_ts_all_with_results

In [ ]:
df_shap_ts_fp = df_shap_ts_all_with_results[(df_shap_ts_all_with_results["Prediction"] == "sepsis") & (
        df_shap_ts_all_with_results["Ground Truth"] == "control")]
df_shap_ts_fp["conf"] = "False Positive"
df_shap_ts_fp

In [ ]:
df_shap_ts_tp = df_shap_ts_all_with_results[
    (df_shap_ts_all_with_results["Prediction"] == "sepsis") & (df_shap_ts_all_with_results["Ground Truth"] == "sepsis")]
df_shap_ts_tp["conf"] = "True Positive"
df_shap_ts_tp

In [ ]:
df_shap_ts_fn = df_shap_ts_all_with_results[(df_shap_ts_all_with_results["Prediction"] == "control") & (
        df_shap_ts_all_with_results["Ground Truth"] == "sepsis")]
df_shap_ts_fn["conf"] = "False Negative"
df_shap_ts_fn

In [ ]:
df_shap_ts_tn = df_shap_ts_all_with_results[(df_shap_ts_all_with_results["Prediction"] == "control") & (
        df_shap_ts_all_with_results["Ground Truth"] == "control")]
df_shap_ts_tn["conf"] = "True Negative"
df_shap_ts_tn

Combining the df back together with the conf case label


In [ ]:
df_shap_ts_all_conf = pd.concat([df_shap_ts_tn, df_shap_ts_fp, df_shap_ts_tp, df_shap_ts_fn])
df_shap_ts_all_conf = df_shap_ts_all_conf.drop(columns=["Ground Truth", "Prediction"])

df_shap_ts_all_conf = VariableNames.get_pretty_dataframe(df_shap_ts_all_conf)

df_means = df_shap_ts_all_conf.groupby("conf").mean()
sorted_features = df_means.mean().sort_values(ascending=False).index

df_shap_ts_all_conf = df_shap_ts_all_conf[[*sorted_features, "conf"]]
# df_shap_ts_all_conf = insert_line_breaks_in_columns(df=df_shap_ts_all_conf, max_length=15)


df_shap_ts_all_conf

In [ ]:
num_features_per_chunk = 15

chunks = [sorted_features[i:i + num_features_per_chunk] for i in range(0, len(sorted_features), num_features_per_chunk)]

chunks = chunks[:4]

dfs_chunked = []

for i in range(0, len(sorted_features), num_features_per_chunk):
    chunk = sorted_features[i:i + num_features_per_chunk]
    dfs_chunked.append(df_shap_ts_all_conf[chunk.tolist() + ['conf']].copy())

In [ ]:
import matplotlib.ticker as ticker
import palettable


def plot_chunked_barplot(df, title):
    # fig, ax = plt.subplots(figsize=(7, 9))  # Create a new figure for each plot
    fig, ax, font_labels, font_annotations = figure_setup(1, 1, width_cm=15, height_cm=14)
    sns.barplot(
        data=df.melt(id_vars=["conf"]),
        x="value",
        y="variable",
        hue="conf",
        ax=ax,
        orient="h",
        # palette=sns.color_palette("magma", n_colors=4),
        # palette=palettable.colorbrewer.qualitative.Set1_4_r.mpl_colors,
        palette=palettable.tableau.PurpleGray_6_r.mpl_colors,
        # capsize=0.05,
        # err_kws={"linewidth": 1.5}
        errorbar=None
    )
    ax.set_title(title, fontsize=10)
    # ax.set_yticklabels(ax.get_yticklabels(), fontsize=6)
    ax.tick_params(axis="y", which="major", pad=1)

    ax.legend(
        title="Confusion Matrix Cases",
        title_fontproperties=font_labels,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.1),
        ncol=4,
        frameon=False,
        fancybox=True,
        shadow=True,
        prop=font_labels,
    )
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.2g}'))
    ax.set_xlabel("Mean Absolute SHAP Value")

    ax.set_ylabel("Time Series Features",
                  )

    ax.margins(x=0)

    plt.tight_layout()
    plt.show()

    return fig, ax


# Plot each chunk in an individual figure
for i, df_chunk in enumerate(dfs_chunked):
    # title = f'Top Features {i * num_features_per_chunk + 1}-{(i + 1) * num_features_per_chunk}'
    title_old = f'Top Features {i * num_features_per_chunk + 1}-{(i + 1) * num_features_per_chunk}'
    title = ""
    fig, ax = plot_chunked_barplot(df_chunk, title)

    if save_as_svg := True:
        fig.savefig(plot_output_dir / Path(f"shap_global_mean_abs_test_{title_old}.svg"), format="svg")



In [ ]:


def plot_chunked_barplot(df, title):
    # fig, ax = plt.subplots(figsize=(7, 9))  # Create a new figure for each plot
    fig, ax, font_labels, font_annotations = figure_setup(1, 1, width_cm=15, height_cm=14)

    color_palette = palettable.tableau.PurpleGray_6_r.mpl_colors[:]
   
    sns.barplot(
        data=df.melt(id_vars=["conf"]),
        x="value",
        y="variable",
        hue="conf",
        ax=ax,
        orient="h",
        # palette=sns.color_palette("magma", n_colors=4),
        # palette=palettable.colorbrewer.qualitative.Set1_4_r.mpl_colors,
        palette=color_palette,
        # capsize=0.05,
        # err_kws={"linewidth": 1.5}
        errorbar=None
    )

    hatches = ['///', '||', '\\\\', 'xxx']  # Hatching for bars
    legend_hatches = ['///', '|||', '\\\\\\', 'xxx']  # Denser hatching for legend

    for i, thisbar in enumerate(ax.patches):
        # Get the hue index (confusion matrix case) for the current bar
        hue_idx = i // len(df.melt(id_vars=["conf"])['variable'].unique()) % len(hatches)
        thisbar.set_hatch(hatches[hue_idx])

    ax.tick_params(axis="y", which="major", pad=1)

    # Create custom legend with denser hatching
    handles = []
    for i, conf in enumerate(df['conf'].unique()):
        handles.append(Patch(
            # facecolor=palettable.tableau.PurpleGray_6_r.mpl_colors[i],
            facecolor=color_palette[i],
            hatch=legend_hatches[i % len(legend_hatches)],
            label=conf))

    ax.set_title(title, fontsize=10)
    # ax.set_yticklabels(ax.get_yticklabels(), fontsize=6)
    ax.tick_params(axis="y", which="major", pad=1)



    ax.legend(
        handles=handles,
        title="Confusion Matrix Cases",
        title_fontproperties=font_labels,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.1),
        ncol=2,
        frameon=False,
        fancybox=True,
        shadow=True,
        prop=font_labels,
        handleheight=2,
        handlelength=3,
    )

    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.2g}'))
    ax.set_xlabel("Mean Absolute SHAP Value")

    ax.set_ylabel("Time Series Features",
                  )

    ax.margins(x=0)

    plt.tight_layout()
    plt.show()

    return fig, ax


# Plot each chunk in an individual figure
for i, df_chunk in enumerate(dfs_chunked):
    # title = f'Top Features {i * num_features_per_chunk + 1}-{(i + 1) * num_features_per_chunk}'
    title_old = f'Top Features {i * num_features_per_chunk + 1}-{(i + 1) * num_features_per_chunk}'
    title = ""
    fig, ax = plot_chunked_barplot(df_chunk, title)

    if save_as_svg := True:
        fig.savefig(plot_output_dir / Path(f"shap_global_mean_abs_test_{title_old}_hatched.svg"), format="svg")



# Static Features Global mean absolute SHAP value

In [ ]:
df_shap_static_gt_control = df_shap_static_all_with_results.loc[
                            df_shap_static_all_with_results["Ground Truth"] == "control",
                            :].drop(columns=["Ground Truth", "Prediction"])

df_shap_static_gt_sepsis = df_shap_static_all_with_results.loc[
                           df_shap_static_all_with_results["Ground Truth"] == "sepsis", :].drop(
    columns=["Ground Truth", "Prediction"])

df_shap_static_pred_control = df_shap_static_all_with_results.loc[
                              df_shap_static_all_with_results["Prediction"] == "control",
                              :].drop(columns=["Ground Truth", "Prediction"])

df_shap_static_pred_sepsis = df_shap_static_all_with_results.loc[
                             df_shap_static_all_with_results["Prediction"] == "sepsis",
                             :].drop(columns=["Ground Truth", "Prediction"])
df_shap_static_all_with_results

In [ ]:

df_shap_static_fp = df_shap_static_all_with_results[(df_shap_static_all_with_results["Prediction"] == "sepsis") & (
        df_shap_static_all_with_results["Ground Truth"] == "control")]
df_shap_static_fp["conf"] = "False Positive"
df_shap_static_fp

In [ ]:

df_shap_static_tp = df_shap_static_all_with_results[
    (df_shap_static_all_with_results["Prediction"] == "sepsis") & (
            df_shap_static_all_with_results["Ground Truth"] == "sepsis")]
df_shap_static_tp["conf"] = "True Positive"
df_shap_static_tp

In [ ]:

df_shap_static_fn = df_shap_static_all_with_results[(df_shap_static_all_with_results["Prediction"] == "control") & (
        df_shap_static_all_with_results["Ground Truth"] == "sepsis")]
df_shap_static_fn["conf"] = "False Negative"
df_shap_static_fn

In [ ]:

df_shap_static_tn = df_shap_static_all_with_results[(df_shap_static_all_with_results["Prediction"] == "control") & (
        df_shap_static_all_with_results["Ground Truth"] == "control")]
df_shap_static_tn["conf"] = "True Negative"
df_shap_static_tn

Combining the df back together with the conf case label

In [ ]:
df_shap_static_all_conf = pd.concat([df_shap_static_tn, df_shap_static_fp, df_shap_static_tp, df_shap_static_fn])
df_shap_static_all_conf = df_shap_static_all_conf.drop(columns=["Ground Truth", "Prediction"])

df_shap_static_all_conf = VariableNames.get_pretty_dataframe(df_shap_static_all_conf)

df_means = df_shap_static_all_conf.groupby("conf").mean()
sorted_features = df_means.mean().sort_values(ascending=False).index

df_shap_static_all_conf = df_shap_static_all_conf[[*sorted_features, "conf"]]
# df_shap_ts_all_conf = insert_line_breaks_in_columns(df=df_shap_ts_all_conf, max_length=15)


df_shap_static_all_conf

In [ ]:
fig, ax, font_labels, font_annotations = figure_setup(1, 1, width_cm=15, height_cm=14)
sns.barplot(
    data=df_shap_static_all_conf.melt(id_vars=["conf"]),
    x="value",
    y="variable",
    hue="conf",
    ax=ax,
    orient="h",
    # palette=sns.color_palette("magma", n_colors=4),
    # palette=palettable.colorbrewer.qualitative.Set1_4_r.mpl_colors,
    palette=palettable.tableau.PurpleGray_6_r.mpl_colors,
    # capsize=0.05,
    # err_kws={"linewidth": 1.5}
    errorbar=None
)
# ax.set_title(title, fontsize=10)
# ax.set_yticklabels(ax.get_yticklabels(), fontsize=6)
ax.tick_params(axis="y", which="major", pad=1)

ax.legend(
    title="Confusion Matrix Cases",
    title_fontproperties=font_labels,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.1),
    ncol=4,
    frameon=False,
    fancybox=True,
    shadow=True,
    prop=font_labels,
    handleheight=2,
    handlelength=3,
)
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.2g}'))
ax.set_xlabel("Mean Absolute SHAP Value")

ax.set_ylabel("Static Features",
              )

ax.margins(x=0)

plt.tight_layout()
plt.show()

if save_as_svg := True:
    fig.savefig(plot_output_dir / Path(f"shap_global_mean_abs_test_static.svg"), format="svg")



In [ ]:
df_shap_static_all_conf.melt(id_vars=["conf"])

In [ ]:

fig, ax, font_labels, font_annotations = figure_setup(1, 1, width_cm=15, height_cm=14)

# color_palette = palettable.matplotlib.Plasma_6.mpl_colors[:]
color_palette = palettable.tableau.PurpleGray_6_r.mpl_colors[:]
# Create the bar plot
barplot = sns.barplot(
    data=df_shap_static_all_conf.melt(id_vars=["conf"]),
    x="value",
    y="variable",
    hue="conf",
    ax=ax,
    orient="h",
    palette=color_palette,
    errorbar=None
)

hatches = ['///', '||', '\\\\', 'xxx']  # Hatching for bars
legend_hatches = ['///', '|||', '\\\\\\', 'xxx']  # Denser hatching for legend

for i, thisbar in enumerate(ax.patches):
    # Get the hue index (confusion matrix case) for the current bar
    hue_idx = i // len(df_shap_static_all_conf.melt(id_vars=["conf"])['variable'].unique()) % len(hatches)
    thisbar.set_hatch(hatches[hue_idx])

ax.tick_params(axis="y", which="major", pad=1)

# Create custom legend with denser hatching
handles = []
for i, conf in enumerate(df_shap_static_all_conf['conf'].unique()):
    handles.append(Patch(
        # facecolor=palettable.tableau.PurpleGray_6_r.mpl_colors[i],
        facecolor=color_palette[i],
        hatch=legend_hatches[i % len(legend_hatches)],
        label=conf))

ax.legend(
    handles=handles,
    title="Confusion Matrix Cases",
    title_fontproperties=font_labels,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.1),
    ncol=2,
    frameon=False,
    fancybox=True,
    shadow=True,
    prop=font_labels,
    handleheight=2,
    handlelength=3,
)

# Format the x-axis labels
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.2g}'))
ax.set_xlabel("Mean Absolute SHAP Value")

# Set the y-axis label
ax.set_ylabel("Static Features")

# Adjust margins and layout
ax.margins(x=0)
plt.tight_layout()

# Display the plot
plt.show()

# Save as SVG if needed
if save_as_svg := True:
    fig.savefig(plot_output_dir / Path(f"shap_global_mean_abs_test_static_hatched_pg.svg"), format="svg")

## Cohort SHAP Feature Importance

In [ ]:
df_test = sepsis_pred_data.df_test

In [ ]:
df_test_unique = df_test.drop_duplicates(subset=[sepsis_pred_data.sample_index_column, "patientunitstayid"])
df_test_unique.head()

In [ ]:
df_test_sample_indices_id = df_test_unique.loc[:, [sepsis_pred_data.sample_index_column, "patientunitstayid"]]
df_test_sample_indices_id.head()

In [ ]:
df_static = sepsis_pred_data.df_static
df_static.head()

In [ ]:
df_static_test = df_static.loc[df_static["patientunitstayid"].isin(df_test_sample_indices_id["patientunitstayid"])]
df_static_test["unittype"].describe()

In [ ]:
df_static_test

In [ ]:
df_static_ts_test = pd.merge(df_test_sample_indices_id, df_static_test, on="patientunitstayid", how="left")

In [ ]:
df_test_cohort_sample_idx = df_static_ts_test.loc[:, [sepsis_pred_data.sample_index_column, "unittype"]]
df_test_cohort_sample_idx["unittype"].value_counts()

In [ ]:
df_static["unittype"].value_counts()

In [ ]:
df = df_test_cohort_sample_idx[df_test_cohort_sample_idx["unittype"] != ""]
# df2 = df_test_cohort_sample_idx[df_test_cohort_sample_idx["unittype"] == crit1eria_2]



### Instance Wise (Temporal SHAP) 

In [ ]:
shap_instance_sample = shap_values_batches[0][0].shape
sample_index = 17


In [ ]:
df_sample_ts = pd.DataFrame(sample_ts_batches[0][sample_index].cpu(), columns=sepsis_pred_data.considered_features)
df_sample_ts = VariableNames.get_pretty_dataframe(df_sample_ts)
df_sample_ts

batch x batch_size x time_steps x features x class

In [ ]:
shap_values_batches[0][0][sample_index].squeeze(2).shape

In [ ]:

shap_explanation_ts_sample = shap.Explanation(shap_values_batches[0][0][sample_index].squeeze(2),
                                              feature_names=df_sample_ts.columns, data=df_sample_ts)

In [ ]:

number_items_to_display = 16


In [ ]:
shap_heatmap_mod(shap_values=shap_explanation_ts_sample, max_display=number_items_to_display)

In [ ]:
sample_importance = np.argsort(shap_explanation_ts_sample.abs.mean(0).values)

sorted_feature_names_ts = [None] * len(df_sample_ts.columns)

for rank, idx in enumerate(sample_importance):
    sorted_feature_names_ts[rank] = df_sample_ts.columns[idx]

heatmap_importance_ts_features = sorted_feature_names_ts[-number_items_to_display:]

df_sample_ts_heatmap = df_sample_ts.loc[:, heatmap_importance_ts_features]

In [ ]:
heat_values, heat_feature_values, heat_feature_names = shap_heatmap_values(shap_values=shap_explanation_ts_sample,
                                                                           max_display=number_items_to_display)
df_heat_diy = pd.DataFrame(data=heat_values, columns=heat_feature_names)
vmin, vmax = np.nanpercentile(heat_values.flatten(), [1, 99])

fig_heat_s, ax_heat_s, font_lb, font_anot = figure_setup(1, num_cols=1, width_cm=12, height_cm=8, num_colors=12,
                                                         color_offset=4, sharex=True)
sns.heatmap(df_heat_diy.T,
            cmap=sns.color_palette("icefire", as_cmap=True),
            ax=ax_heat_s, cbar=True,
            vmin=min(vmin, -vmax),
            vmax=max(-vmin, vmax),
            alpha=0.95,
            )

# Invert the y-axis to match the heatmap coordinates
ax_heat_s.invert_yaxis()

for i, feature_name in enumerate(heat_feature_names):
    if i == len(heat_feature_names) - 1:
        break
    center_y = i
    ax_heat_s.plot(df_sample_ts_heatmap.index,
                   # ax_heat_s.scatter(df_sample_ts_heatmap.index,
                   # df_sample_ts_heatmap[feature_name] / df_sample_ts_heatmap[feature_name].max() + center_y,
                   df_sample_ts_heatmap[feature_name] / 1 + center_y,
                   label=feature_name,
                   color="magenta"
                   )

ax_heat_s.set_xlabel("Time [h]", fontproperties=font_lb)
ax_heat_s.set_ylabel("Time Series Features", fontproperties=font_lb)

ax_heat_s.collections[0].colorbar.set_label("SHAP Feature Importance", fontproperties=font_lb)

five_min_hour_formatter = ticker.FuncFormatter(func=five_min_to_hours)
ax_heat_s.xaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
ax_heat_s.xaxis.set_major_formatter(five_min_hour_formatter)

for tick in ax_heat_s.get_xticklabels() + ax_heat_s.get_yticklabels():
    tick.set_fontproperties(font_anot)



ax_heat_s.set_xticklabels(ax_heat_s.get_xticklabels(), rotation=0)

# Adding legend for line plots
ax_heat_s.legend(
    [],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.1),
    ncol=3,
    frameon=False,
    fancybox=True,
    shadow=True,
    prop=font_anot
)

plt.grid()
plt.tight_layout()
plt.show()

if save_as_svg := True:
    fig_heat_s.savefig(plot_output_dir / Path(f"time_series_shap_sample_{sample_index}.svg"), format="svg")
    fig_heat_s.savefig(plot_output_dir / Path(f"time_series_shap_sample_{sample_index}.png"), format="png", dpi=300)

# global averages bar plot for time series